In [15]:
from langchain_core.prompts import PromptTemplate

In [51]:
template = """
You are an expert resume reviewer and career advisor.
Analyze the following resume vs job description data and give actionable improvement suggestions.


DATA:
- Final Match Score: {final_score}
- Interpretation: {interpretation}
- Phrase Score: {phrase_score}
- Keyword Score: {keyword_score}

- Matched Skills: {matched_skills}
- Missing Skills: {missing_skills}

- Resume Keywords: {resume_keywords}
- Job Description Keywords: {jd_keywords}


TASK:
1. Provide a detailed explanation of the final match score, explaining how well the resume aligns with the job description based on the interpretation.
2. Highlight the key weaknesses in the resume by analyzing the matched and missing skills.
3. Suggest specific, actionable improvements to enhance the resume, especially in terms of required skills, projects, or experience.
4. Recommend important skills that the candidate should focus on learning, based on the identified gaps.
5. Offer suggestions to improve the wording, clarity, and overall impact of the resume, including use of strong action verbs and measurable achievements.
6. Conclude with a clear and concise final verdict on the candidate’s suitability for the role.


OUTPUT FORMAT:
- Score Explanation:
- Weaknesses:
- Improvements:
- Recommended Skills:
- Resume Enhancement Tips:
- Final Verdict:


RESPONSE GUIDELINES:
- Be clear, concise, and professional  
- Avoid generic advice; make suggestions specific to the provided data  
- Focus on practical and actionable recommendations  
- Do not repeat the input data unnecessarily  
"""

In [52]:
prompt = PromptTemplate(
    input_variables=[
        "final_score",
        "interpretation",
        "phrase_score",
        "keyword_score",
        "matched_skills",
        "missing_skills",
        "resume_keywords",
        "jd_keywords"
    ],
    template=template
)

In [18]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    temperature=0.3,
    model_name="llama-3.3-70b-versatile"
)

In [1]:
from parser import process_file
from scorer import Scorer
from embedder import Embedder

c:\Users\sayan\OneDrive\Desktop\Learning\virtusa\projects\resume-analyser\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [19]:
import os

BASE_DIR = os.getcwd()

resume_path = os.path.join(BASE_DIR, "data", "resume2.pdf")
jd_path = os.path.join(BASE_DIR, "data", "sample_jd.txt")

In [3]:
scorer = Scorer()
embedder = Embedder()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1806.93it/s]


In [22]:
import pdfplumber

def extract_text_from_pdf(file_path):
    text = ""

    with pdfplumber.open(file_path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"

    return text

In [23]:
from parser import read_txt, clean_text

In [24]:
resume_text = clean_text(extract_text_from_pdf(resume_path))
jd_text = clean_text(read_txt(jd_path))

In [26]:
from preprocessor import extract_keywords, extract_phrases

resume_keywords = extract_keywords(resume_text)
jd_keywords = extract_keywords(jd_text)

resume_phrases = extract_phrases(resume_text)
jd_phrases = extract_phrases(jd_text)

In [28]:
resume_embeddings = embedder.encode_batch(resume_phrases)
jd_embeddings = embedder.encode_batch(jd_phrases)

In [30]:
phrase_score = scorer.get_score(resume_embeddings, jd_embeddings)
keyword_score = scorer.keyword_score(resume_keywords, jd_keywords)

final_score = scorer.final_score(phrase_score, keyword_score)

In [31]:
matched_skills = resume_keywords & jd_keywords
missing_skills = jd_keywords - resume_keywords

In [53]:
prompt = template.format(
    final_score=round(final_score * 100 , 2),
    interpretation=scorer.interpret(final_score),
    phrase_score=round(phrase_score * 100 , 2),
    keyword_score=round(keyword_score * 100, 2),
    matched_skills=matched_skills,
    missing_skills=missing_skills,
    resume_keywords=resume_keywords,
    jd_keywords=jd_keywords
)

In [37]:
print(f"Phrase score: {phrase_score * 100 :.2f}")
print(f"Keyword score: {keyword_score * 100 :.2f}")
print(f"Final score: {final_score * 100 :.2f}")

Phrase score: 54.22
Keyword score: 62.93
Final score: 56.83


In [54]:
response = llm.invoke(prompt)
print(response.content)

Score Explanation:
The final match score of 56.83 indicates a moderate match between the resume and the job description. This suggests that while the candidate has some relevant skills and experience, there are areas where they do not align with the job requirements. The phrase score of 54.22 and keyword score of 62.93 indicate that the candidate's resume could be improved in terms of using relevant phrases and keywords to describe their skills and experience.

Weaknesses:
The matched skills show that the candidate has a strong foundation in technical skills such as machine learning, natural language processing, and programming languages like Python and JavaScript. However, the missing skills highlight gaps in areas like frontend development, debugging, and testing. The candidate also lacks experience in working with certain technologies like Docker, AWS, and MongoDB.

Improvements:
To enhance the resume, the candidate should focus on adding relevant projects or experience that demonst

In [50]:
print(response.content)

**Resume Analysis and Improvement Suggestions**

### 1. Final Score Explanation

The final match score is 56.83, indicating a moderate match between the resume and the job description. This suggests that while the resume shares some relevant skills and keywords with the job description, there are areas for improvement to increase the match score.

### 2. Key Weaknesses in the Resume

Based on the matched and missing skills, the key weaknesses in the resume are:

* Lack of frontend development skills
* Insufficient mention of responsibility, concept, integration, and debug skills
* Limited exposure to foundation, seamless, and maintainable skills
* Missing keywords such as "profile", "work", "fresher", and "performance"

### 3. Specific Improvement Suggestions

To improve the resume, consider the following:

* Add relevant frontend development skills, such as React or Angular
* Emphasize responsibility, concept, integration, and debug skills in project descriptions
* Highlight experienc